# AIES Lab Task: Security Patrol Agents

> Add blockquote



A security robot patrols two zones, A and B. Each zone is either Safe or contains an Intrusion. The robot initially
knows only its location and must Scan to observe the current zone’s condition.

Implement the following actions:

Action: Purpose
Scan: Observe the current zone’s condition.
RaiseAlarm: Report an intrusion without removing it.
MoveToA: Move from B to A.
MoveToB: Move from A to B.
Wait: Stay in the current zone for one step.
EndPatrol: Finish the inspection.


1. Table-Driven Agent
Implement a lookup table indexed by the complete percept sequence for this initial state: Robot at A, A =
Safe, B = Intrusion. The agent must inspect A, move to B, inspect B, report the intrusion, and end the
patrol.

2. Simple Reflex Agent  
Implement these rules using only the current percept:
Unknown condition: Scan.
Intrusion detected: RaiseAlarm.
A is Safe: MoveToB.
B is Safe: MoveToA.

3. Model-Based Reflex Agent  
Maintain memory of inspected zones and reported intrusions. Inspect both zones, report each intrusion
once, and select EndPatrol when both zones have been inspected and all detected intrusions reported.
Access zone conditions only through Scan.

Print each percept and selected action. Stop after eight actions or EndPatrol, whichever occurs first.

Instructions:
Zone conditions remain unchanged: A Safe zone stays Safe, and an Intrusion stays present throughout the
simulation.
Percept = (location, observed_condition): For example, (A, Safe) means the robot is in A and has observed that A
is Safe.
Initially Unknown: The robot starts with (A, Unknown) because it has not scanned A yet.
Scan updates the observation: Scanning A changes its percept to (A, Safe) if A is Safe.
Movement resets the observation: Moving to B produces (B, Unknown). The robot must scan B to discover its
condition.
Other actions preserve the observation: RaiseAlarm and Wait do not change the observed condition. Raising an
alarm does not remove the intrusion.
Wait is optional: You should implement it, but the agents do not have to use it.


# Table-Driven Agent

In [51]:
table = {
    (('A', 'Unknown'),): 'Scan',
    (('A',  'Unknown'), ('A', 'Safe')):  'MoveToB',
    (('A', 'Unknown'), ('A', 'Safe'),  ('B',  'Unknown')): 'Scan',
    (('A',  'Unknown'), ('A',  'Safe'), ('B', 'Unknown'), ('B', 'Intrusion')): 'RaiseAlarm',
    (('A', 'Unknown'), ('A', 'Safe'),  ('B',  'Unknown'), ('B', 'Intrusion'), ('B', 'Intrusion')):  'EndPatrol'
}

In [52]:
percepts = []

def table_driven_agent(percept):
    percepts.append(percept)
    action = table.get(tuple(percepts),  'EndPatrol')
    return action


In [53]:
real_world = {'A': 'Safe', 'B': 'Intrusion'}
location = 'A'
observed_condition = 'Unknown'

print("--------Table-Driven Agent ---------")
for step in range(8):
    percept = (location, observed_condition)
    print('Perception Received:', percept)

    action = table_driven_agent(percept)
    print('Action Selected:', action)

    if action == 'EndPatrol':
        break
    elif action ==  'Scan':
        observed_condition = real_world[location]
    elif action == 'MoveToA':
        location = 'A'
        observed_condition = 'Unknown'
    elif action == 'MoveToB':
        location =  'B'
        observed_condition = 'Unknown'

--------Table-Driven Agent ---------
Perception Received: ('A', 'Unknown')
Action Selected: Scan
Perception Received: ('A', 'Safe')
Action Selected: MoveToB
Perception Received: ('B', 'Unknown')
Action Selected: Scan
Perception Received: ('B', 'Intrusion')
Action Selected: RaiseAlarm
Perception Received: ('B', 'Intrusion')
Action Selected: EndPatrol


# Simple Reflex Agent

In [60]:
def simple_reflex_agent(percept):
    location = percept[0]
    condition = percept[1]

    if condition == 'Unknown':
        action =  'Scan'
    elif condition == 'Intrusion':
        action = 'RaiseAlarm'
    elif location == 'A' and  condition == 'Safe':
        action = 'MoveToB'
    elif location == 'B' and condition == 'Safe':
        action = 'MoveToA'
    else:
        action = 'Wait'

    return action


In [61]:
real_world = {'A': 'Safe', 'B': 'Intrusion'}
location = 'A'
observed_condition = 'Unknown'

print("--------- Simple Reflex Agent -------------")
for step in range(8):
    percept = (location, observed_condition)
    print('Perception Received:', percept)

    action = simple_reflex_agent(percept)
    print('Action Selected:', action)

    if action == 'EndPatrol':
        break
    elif action == 'Scan':
        observed_condition = real_world[location]
    elif action == 'MoveToA':
        location = 'A'
        observed_condition = 'Unknown'
    elif action == 'MoveToB':
        location = 'B'
        observed_condition = 'Unknown'

--------- Simple Reflex Agent -------------
Perception Received: ('A', 'Unknown')
Action Selected: Scan
Perception Received: ('A', 'Safe')
Action Selected: MoveToB
Perception Received: ('B', 'Unknown')
Action Selected: Scan
Perception Received: ('B', 'Intrusion')
Action Selected: RaiseAlarm
Perception Received: ('B', 'Intrusion')
Action Selected: RaiseAlarm
Perception Received: ('B', 'Intrusion')
Action Selected: RaiseAlarm
Perception Received: ('B', 'Intrusion')
Action Selected: RaiseAlarm
Perception Received: ('B', 'Intrusion')
Action Selected: RaiseAlarm


# Model-Based Reflex Agent

In [65]:
model = {'A': 'Unknown',  'B': 'Unknown'}

def model_based_reflex_agent(percept):
    global model
    location = percept[0]
    condition = percept[1]

    if condition != 'Unknown' and  model[location] != 'Reported':
        model[location] =  condition

    if model['A'] in ['Safe', 'Reported'] and  model['B'] in ['Safe', 'Reported']:
        action = 'EndPatrol'
    elif condition == 'Unknown':
        action =  'Scan'
    elif model[location] == 'Intrusion':
        action = 'RaiseAlarm'
    elif location == 'A':
        action = 'MoveToB'
    elif location == 'B':
        action = 'MoveToA'
    else:
        action =  'Wait'

    if action ==  'RaiseAlarm':
        model[location] =  'Reported'

    return action


In [66]:
real_world = {'A': 'Safe', 'B': 'Intrusion'}
location = 'A'
observed_condition = 'Unknown'

print("------- Model-Based Reflex Agent Simulation ----------")
for step in range(8):
    percept = (location, observed_condition)
    print('Perception Received:', percept)

    action = model_based_reflex_agent(percept)
    print('Action Selected:', action)
    print('Current Model:', model)
    print('-----------------------------------------------------')

    if action == 'EndPatrol':
        break
    elif action == 'Scan':
        observed_condition = real_world[location]
    elif action == 'MoveToA':
        location = 'A'
        observed_condition = 'Unknown'
    elif action == 'MoveToB':
        location = 'B'
        observed_condition = 'Unknown'

------- Model-Based Reflex Agent Simulation ----------
Perception Received: ('A', 'Unknown')
Action Selected: Scan
Current Model: {'A': 'Unknown', 'B': 'Unknown'}
-----------------------------------------------------
Perception Received: ('A', 'Safe')
Action Selected: MoveToB
Current Model: {'A': 'Safe', 'B': 'Unknown'}
-----------------------------------------------------
Perception Received: ('B', 'Unknown')
Action Selected: Scan
Current Model: {'A': 'Safe', 'B': 'Unknown'}
-----------------------------------------------------
Perception Received: ('B', 'Intrusion')
Action Selected: RaiseAlarm
Current Model: {'A': 'Safe', 'B': 'Reported'}
-----------------------------------------------------
Perception Received: ('B', 'Intrusion')
Action Selected: EndPatrol
Current Model: {'A': 'Safe', 'B': 'Reported'}
-----------------------------------------------------
